In [5]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from catboost import CatBoostClassifier

# --- CONFIGURATION ---
DATA_FILE = "../data/processed/bio_markers_refined.csv"
MODEL_SAVE_PATH = "../mlModels/Cry/best_cry_model.pkl"

def train_best():
    print("⏳ Loading Data...")
    if not os.path.exists(DATA_FILE):
        print("❌ CSV not found. Run extract_biomarkers.py first!")
        return

    df = pd.read_csv(DATA_FILE)
    df.fillna(0, inplace=True)
    
    X = df.drop('label', axis=1)
    y = df['label']
    
    print(f"✅ Data Loaded: {len(X)} samples with {len(X.columns)} features.")

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # --- DEFINE THE GRID SEARCH ---
    # CatBoost's native grid search
    param_grid = {
        'iterations': [1000],
        'learning_rate': [0.03, 0.05],
        'depth': [6, 8, 10],
        'l2_leaf_reg': [3, 5]
    }

    print("🚀 Starting Grid Search (This might take 5-10 minutes)...")
    
    model = CatBoostClassifier(
        loss_function='MultiClass', 
        verbose=0,
        task_type="CPU"
    )

    # Use CatBoost's grid_search method
    grid_search_result = model.grid_search(
        param_grid,
        X=X_train,
        y=y_train,
        cv=3,
        verbose=False,
        plot=False,
        stratified=True
    )

    # --- RESULTS ---
    print(f"\n✅ Best Settings Found: {grid_search_result['params']}")
    
    # Train final model with best parameters
    best_model = CatBoostClassifier(
        **grid_search_result['params'],
        loss_function='MultiClass',
        verbose=0,
        task_type="CPU"
    )
    best_model.fit(X_train, y_train, verbose=False)

    # Final Evaluation
    y_pred = best_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n🏆 ULTIMATE ACCURACY: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=['Hunger', 'Pain', 'Normal']))

    # Save
    joblib.dump(best_model, MODEL_SAVE_PATH)
    print(f"💾 Best Model Saved to {MODEL_SAVE_PATH}")

if __name__ == "__main__":
    train_best()


⏳ Loading Data...
✅ Data Loaded: 1095 samples with 231 features.
🚀 Starting Grid Search (This might take 5-10 minutes)...

bestTest = 1.046836205
bestIteration = 86


bestTest = 1.04989548
bestIteration = 67


bestTest = 1.050326713
bestIteration = 136


bestTest = 1.04898824
bestIteration = 64


bestTest = 1.069436562
bestIteration = 46


bestTest = 1.073596923
bestIteration = 45


bestTest = 1.071578299
bestIteration = 84


bestTest = 1.067155699
bestIteration = 46


bestTest = 1.088410226
bestIteration = 28


bestTest = 1.089650001
bestIteration = 16


bestTest = 1.086563107
bestIteration = 29


bestTest = 1.085144186
bestIteration = 26

Training on fold [0/3]

bestTest = 1.056386632
bestIteration = 82

Training on fold [1/3]

bestTest = 1.044735843
bestIteration = 81

Training on fold [2/3]

bestTest = 1.059069545
bestIteration = 57


✅ Best Settings Found: {'depth': 6, 'learning_rate': 0.03, 'l2_leaf_reg': 3, 'iterations': 1000}

🏆 ULTIMATE ACCURACY: 26.03%
              precision